In [ ]:
# 1. Import libraries
import pandas as pd              # pandas = used to work with tables (dataframes)
import numpy as np               # numpy = used for numbers and arrays
import matplotlib.pyplot as plt  # matplotlib = used to create graphs

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV  # split data + tuning models
from sklearn.preprocessing import StandardScaler                    # scale/normalize data

In [ ]:
# Models
from sklearn.linear_model import LogisticRegression     # Logistic Regression model
from sklearn.tree import DecisionTreeClassifier         # Decision Tree model
from sklearn.neighbors import KNeighborsClassifier      # KNN model

In [ ]:
# Evaluation
from sklearn.metrics import (
    accuracy_score,             # checks how many predictions are correct
    classification_report,     # detailed report (precision, recall, etc.)
    confusion_matrix,          # table of correct vs wrong predictions
    ConfusionMatrixDisplay,    # display confusion matrix as graph
    roc_curve,                 # ROC curve values
    roc_auc_score              # AUC score (model performance)
)

In [ ]:
# 2. Load dataset
df = pd.read_csv("bank-additional-full.csv", sep=";")  # load CSV file with ; separator

In [ ]:
# 🔍 Check columns (ADD HERE)
print(df.columns)   # print all column names

In [ ]:
# 3. Dataset Exploration
print("First 5 rows:")
print(df.head())    # show first 5 rows of data

In [ ]:
print("\nDataset Info:")
print(df.info())    # show data types and missing values

In [ ]:
print("\nStatistics:")
print(df.describe())  # show statistics like mean, min, max

In [ ]:
# 4. Handle missing values
df = df.fillna(df.median(numeric_only=True))  # fill missing numeric values with median

In [ ]:
# 5. Separate features and target
# Target variable (yes/no → 1/0)
y = df["y"].map({"yes": 1, "no": 0})  # convert yes/no into numbers

In [ ]:
# Features
X = df.drop("y", axis=1)  # remove target column from features

In [ ]:
# Convert categorical features to numeric
X = pd.get_dummies(X, drop_first=True)  # convert text columns into numbers

In [ ]:
# 6. Train-test split
# Using 80/20 train-test split and 3-fold cross-validation (GridSearchCV) for tuning
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42   # 80% train, 20% test
)

In [ ]:
# 7. Normalization
scaler = StandardScaler()                  # create scaler
X_train = scaler.fit_transform(X_train)    # fit and transform training data
X_test = scaler.transform(X_test)          # only transform test data

=========================
BEFORE TUNING (DEFAULT MODELS)
=========================

In [ ]:
# Logistic Regression BEFORE tuning
model1_default = LogisticRegression(max_iter=1000)   # create model
model1_default.fit(X_train, y_train)                 # train model
y_pred1_default = model1_default.predict(X_test)     # predict test data

In [ ]:
print("\nLogistic Regression BEFORE tuning:", accuracy_score(y_test, y_pred1_default))  # print accuracy

In [ ]:
# Decision Tree BEFORE tuning
model2_default = DecisionTreeClassifier(random_state=42)  # create model
model2_default.fit(X_train, y_train)                      # train model
y_pred2_default = model2_default.predict(X_test)          # predict

In [ ]:
print("Decision Tree BEFORE tuning:", accuracy_score(y_test, y_pred2_default))  # accuracy

In [ ]:
# KNN BEFORE tuning
model3_default = KNeighborsClassifier()   # create KNN model
model3_default.fit(X_train, y_train)      # train model
y_pred3_default = model3_default.predict(X_test)  # predict

In [ ]:
print("KNN BEFORE tuning:", accuracy_score(y_test, y_pred3_default))  # accuracy

=========================
MODEL 1: Logistic Regression (with tuning)
=========================

In [ ]:
param_grid_lr = {
    "C": [0.001, 0.01, 0.1, 1, 10, 100],     # regularization strength
    "solver": ["liblinear"],                 # algorithm used
    "class_weight": [None, "balanced"]       # handle imbalance
}

In [ ]:
grid_lr = GridSearchCV(
    LogisticRegression(max_iter=1000, class_weight="balanced"),  # base model
    param_grid_lr,     # parameters to test
    cv=3,              # 3-fold cross validation
    scoring="accuracy" # optimize accuracy
)

In [ ]:
grid_lr.fit(X_train, y_train)   # train with different parameter combinations

In [ ]:
model1 = grid_lr.best_estimator_  # best model found

In [ ]:
print("\nBest Logistic Regression Parameters:", grid_lr.best_params_)  # show best params

In [ ]:
y_pred1 = model1.predict(X_test)  # predict using best model

=========================
MODEL 2: Decision Tree (with tuning)
=========================

In [ ]:
param_grid = {
    "max_depth": [5, 10, 15, 20, None],   # tree depth
    "min_samples_split": [2, 5, 10],      # split condition
    "min_samples_leaf": [1, 2, 5, 10],    # leaf size
    "criterion": ["gini", "entropy"],     # split method
    "max_features": ["sqrt", "log2", None]  # features used
}

In [ ]:
grid = GridSearchCV(
    DecisionTreeClassifier(class_weight=None, random_state=42),
    param_grid,
    cv=3,
    scoring="accuracy"
)

In [ ]:
grid.fit(X_train, y_train)   # train

In [ ]:
model2 = grid.best_estimator_  # best tree

In [ ]:
print("\nBest Decision Tree Parameters:", grid.best_params_)

In [ ]:
y_pred2 = model2.predict(X_test)  # predict

=========================
MODEL 3: KNN (with tuning)
=========================

In [ ]:
param_grid_knn = {
    "n_neighbors": [3, 5, 7, 9]   # number of neighbors
}

In [ ]:
grid_knn = GridSearchCV(
    KNeighborsClassifier(),
    param_grid_knn,
    cv=3,
    scoring="accuracy"
)

In [ ]:
grid_knn.fit(X_train, y_train)  # train

In [ ]:
model3 = grid_knn.best_estimator_  # best KNN

In [ ]:
print("\nBest KNN Parameters:", grid_knn.best_params_)

In [ ]:
y_pred3 = model3.predict(X_test)  # predict

=========================
BEFORE vs AFTER (TABLE + CHART)
=========================

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score  # evaluation metrics

In [ ]:
def get_metrics(y_true, y_pred, model, X):
    return [
        accuracy_score(y_true, y_pred),                        # accuracy
        precision_score(y_true, y_pred),                       # precision
        recall_score(y_true, y_pred),                          # recall
        f1_score(y_true, y_pred),                              # f1 score
        roc_auc_score(y_true, model.predict_proba(X)[:, 1])    # AUC score
    ]

In [ ]:
rows = []  # store results

In [ ]:
def add_row(name, stage, values):
    rows.append([
        name, stage,
        round(values[0], 4),  # accuracy
        round(values[1], 4),  # precision
        round(values[2], 4),  # recall
        round(values[3], 4),  # f1
        round(values[4], 4)   # AUC
    ])

In [ ]:
# Logistic Regression
add_row("Logistic Regression", "Before", get_metrics(y_test, y_pred1_default, model1_default, X_test))
add_row("Logistic Regression", "After", get_metrics(y_test, y_pred1, model1, X_test))

In [ ]:
# Decision Tree
add_row("Decision Tree", "Before", get_metrics(y_test, y_pred2_default, model2_default, X_test))
add_row("Decision Tree", "After", get_metrics(y_test, y_pred2, model2, X_test))

In [ ]:
# KNN
add_row("KNN", "Before", get_metrics(y_test, y_pred3_default, model3_default, X_test))
add_row("KNN", "After", get_metrics(y_test, y_pred3, model3, X_test))

=========================
PRINT TABLE
=========================

In [ ]:
print("\n=== BEFORE vs AFTER TUNING COMPARISON ===\n")
print("Model                Stage   Accuracy  Precision  Recall  F1-score  AUC")
print("----------------------------------------------------------------------")

In [ ]:
for r in rows:
    print(f"{r[0]:<20} {r[1]:<7} {r[2]:<9} {r[3]:<10} {r[4]:<7} {r[5]:<9} {r[6]}")  # print formatted table

=========================
BAR CHART
=========================

In [ ]:
metrics = ["Accuracy", "Precision", "Recall", "F1-score", "AUC"]  # metric names

In [ ]:
for model_name in ["Logistic Regression", "Decision Tree", "KNN"]:
    model_rows = [r for r in rows if r[0] == model_name]  # filter rows

    before = model_rows[0][2:]  # before tuning values
    after = model_rows[1][2:]   # after tuning values

    x = np.arange(len(metrics))  # x positions

    plt.figure()
    plt.bar(x - 0.2, before, width=0.4, label="Before")  # before bars
    plt.bar(x + 0.2, after, width=0.4, label="After")    # after bars

    plt.xticks(x, metrics)  # x labels
    plt.title(f"{model_name}: Before vs After Tuning")
    plt.ylabel("Score")
    plt.legend()
    plt.show()

=========================
EVALUATION
=========================

In [ ]:
print("\n=== Model Comparison (Accuracy) ===")
print("Logistic Regression:", accuracy_score(y_test, y_pred1))
print("Decision Tree:", accuracy_score(y_test, y_pred2))
print("KNN:", accuracy_score(y_test, y_pred3))

In [ ]:
# Detailed report
print("\n=== Classification Report (Logistic Regression) ===")
print(classification_report(y_test, y_pred1))

In [ ]:
print("\n=== Classification Report (Decision Tree) ===")
print(classification_report(y_test, y_pred2))

In [ ]:
print("\n=== Classification Report (KNN) ===")
print(classification_report(y_test, y_pred3))

=========================
📊 VISUALIZATION (ADD HERE)
=========================

In [ ]:
# Accuracy bar chart
models = ["Logistic Regression", "Decision Tree", "KNN"]
accuracies = [
    accuracy_score(y_test, y_pred1),
    accuracy_score(y_test, y_pred2),
    accuracy_score(y_test, y_pred3)
]

In [ ]:
plt.figure()

In [ ]:
colors = ["blue", "orange", "green"]  # colors for bars

In [ ]:
plt.bar(models, accuracies, color=colors)  # plot bars

In [ ]:
plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy")
plt.xlabel("Models")

In [ ]:
plt.show()

In [ ]:
# Precision / Recall / F1 chart
from sklearn.metrics import precision_score, recall_score, f1_score

In [ ]:
precision = [
    precision_score(y_test, y_pred1),
    precision_score(y_test, y_pred2),
    precision_score(y_test, y_pred3)
]

In [ ]:
recall = [
    recall_score(y_test, y_pred1),
    recall_score(y_test, y_pred2),
    recall_score(y_test, y_pred3)
]

In [ ]:
f1 = [
    f1_score(y_test, y_pred1),
    f1_score(y_test, y_pred2),
    f1_score(y_test, y_pred3)
]

In [ ]:
x = np.arange(len(models))  # x positions

In [ ]:
plt.figure()
plt.bar(x - 0.2, precision, width=0.2, label="Precision")
plt.bar(x, recall, width=0.2, label="Recall")
plt.bar(x + 0.2, f1, width=0.2, label="F1-score")

In [ ]:
plt.xticks(x, models)
plt.title("Precision, Recall, F1 Comparison")

In [ ]:
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))  # move legend outside

In [ ]:
plt.tight_layout()  # adjust layout
plt.show()

=========================
CONFUSION MATRICES (ALL MODELS)
=========================

In [ ]:
# Logistic Regression
ConfusionMatrixDisplay.from_predictions(y_test, y_pred1)  # show confusion matrix
plt.title("Confusion Matrix - Logistic Regression")
plt.show()

In [ ]:
# Decision Tree
ConfusionMatrixDisplay.from_predictions(y_test, y_pred2)
plt.title("Confusion Matrix - Decision Tree")
plt.show()

In [ ]:
# KNN
ConfusionMatrixDisplay.from_predictions(y_test, y_pred3)
plt.title("Confusion Matrix - KNN")
plt.show()

=========================
ROC CURVE (ALL MODELS)
=========================

In [ ]:
# Get probabilities
y_prob1 = model1.predict_proba(X_test)[:, 1]  # probabilities for class 1
y_prob2 = model2.predict_proba(X_test)[:, 1]
y_prob3 = model3.predict_proba(X_test)[:, 1]

In [ ]:
# Compute ROC
fpr1, tpr1, _ = roc_curve(y_test, y_prob1)  # false positive rate, true positive rate
fpr2, tpr2, _ = roc_curve(y_test, y_prob2)
fpr3, tpr3, _ = roc_curve(y_test, y_prob3)

In [ ]:
# Compute AUC
auc1 = roc_auc_score(y_test, y_prob1)
auc2 = roc_auc_score(y_test, y_prob2)
auc3 = roc_auc_score(y_test, y_prob3)

In [ ]:
# Plot all together
plt.plot(fpr1, tpr1, label="Logistic Regression (AUC=" + str(round(auc1,3)) + ")")
plt.plot(fpr2, tpr2, label="Decision Tree (AUC=" + str(round(auc2,3)) + ")")
plt.plot(fpr3, tpr3, label="KNN (AUC=" + str(round(auc3,3)) + ")")

In [ ]:
plt.xlabel("False Positive Rate")   # x-axis
plt.ylabel("True Positive Rate")    # y-axis
plt.title("ROC Curve Comparison (All Models)")
plt.legend()
plt.show()